In [22]:
!pip install fastapi uvicorn langchain langchain-community langchain-openai faiss-cpu pydantic pypdf nest-asyncio


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
!pip install --upgrade python-multipart


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Set Your OpenAI API Key

In [24]:
import os

# Replace with your actual OpenAI API key
os.environ["OPENAI_API_KEY"] = "your_actual_openai_api_key_here"

Run the FastAPI Application Code

In [25]:
from typing import List
from fastapi import FastAPI, File, HTTPException, UploadFile, status
from pydantic import BaseModel
import nest_asyncio
import uvicorn

# LangChain Imports
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Enable async loop inside Jupyter
nest_asyncio.apply()

app = FastAPI(
    title="RAG Document Processing API",
    description="Upload documents (PDF, TXT), chunk text with metadata, and query via LLM.",
    version="1.0.0",
)

# Configuration Setup
FAISS_INDEX_PATH = "faiss_index"
UPLOAD_DIR = "uploaded_docs"
os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(FAISS_INDEX_PATH, exist_ok=True)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)


class QueryRequest(BaseModel):
  question: str
  top_k: int = 4


class QueryResponse(BaseModel):
  answer: str
  sources: List[str]


@app.post("/upload", status_code=status.HTTP_201_CREATED)
async def upload_document(file: UploadFile = File(...)):
  """Uploads a PDF or TXT file, extracts text, chunks it, adds metadata, and indexes into FAISS."""
  if not file.filename:
    raise HTTPException(
        status_code=400, detail="Invalid filename or empty upload."
    )

  file_ext = os.path.splitext(file.filename)[1].lower()
  file_path = os.path.join(UPLOAD_DIR, file.filename)

  # Save uploaded file locally
  try:
    with open(file_path, "wb") as buffer:
      content = await file.read()
      if len(content) == 0:
        raise HTTPException(
            status_code=400, detail="Uploaded file is empty (0 bytes)."
        )
      buffer.write(content)
  except Exception as e:
    raise HTTPException(
        status_code=500, detail=f"Failed to save file: {str(e)}"
    )

  # Load and process document based on extension
  try:
    if file_ext == ".pdf":
      loader = PyPDFLoader(file_path)
      docs = loader.load()
    elif file_ext == ".txt":
      loader = TextLoader(file_path, encoding="utf-8")
      docs = loader.load()
    else:
      if os.path.exists(file_path):
        os.remove(file_path)
      raise HTTPException(
          status_code=400,
          detail=(
              "Unsupported file format. Only .pdf and .txt files are allowed."
          ),
      )

    # Attach Metadata explicitly
    for doc in docs:
      doc.metadata.update(
          {"source_file": file.filename, "file_type": file_ext}
      )

    # Chunking
    chunks = text_splitter.split_documents(docs)
    if not chunks:
      raise HTTPException(
          status_code=400, detail="Document yielded no text chunks."
      )

    # Vector Store Management (FAISS)
    if os.path.exists(os.path.join(FAISS_INDEX_PATH, "index.faiss")):
      vector_store = FAISS.load_local(
          FAISS_INDEX_PATH, embeddings, allow_dangerous_deserialization=True
      )
      vector_store.add_documents(chunks)
    else:
      vector_store = FAISS.from_documents(chunks, embeddings)

    vector_store.save_local(FAISS_INDEX_PATH)

    return {
        "filename": file.filename,
        "status": "Success",
        "chunks_indexed": len(chunks),
        "message": "Document processed, chunked, and indexed successfully.",
    }

  except Exception as e:
    if isinstance(e, HTTPException):
      raise e
    raise HTTPException(
        status_code=500, detail=f"Document processing error: {str(e)}"
    )


@app.post("/query", response_model=QueryResponse)
async def query_rag(request: QueryRequest):
  """Accepts user questions, retrieves relevant chunks, and generates a grounded LLM answer."""
  if not request.question.strip():
    raise HTTPException(status_code=400, detail="Query string cannot be empty.")

  if not os.path.exists(os.path.join(FAISS_INDEX_PATH, "index.faiss")):
    raise HTTPException(
        status_code=404,
        detail=(
            "No vector index found. Please upload documents via /upload first."
        ),
    )

  try:
    vector_store = FAISS.load_local(
        FAISS_INDEX_PATH, embeddings, allow_dangerous_deserialization=True
    )
    retriever = vector_store.as_retriever(
        search_kwargs={"k": request.top_k}
    )
    retrieved_docs = retriever.invoke(request.question)

    # Build context and sources
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    sources = list(
        set([doc.metadata.get("source_file", "Unknown") for doc in retrieved_docs])
    )

    # Generation via LLM
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
    prompt_template = ChatPromptTemplate.from_messages([
        (
            "system",
            (
                "You are a precise technical assistant. Answer the user's"
                " question strictly using only the provided context elements."
                " If the answer cannot be found in the context, state 'I cannot"
                " find the answer in the provided documents.'"
            ),
        ),
        (
            "human",
            "Context:\n{context}\n\nQuestion: {question}",
        ),
    ])

    chain = prompt_template | llm
    response = chain.invoke({"context": context, "question": request.question})

    return QueryResponse(answer=response.content, sources=sources)

  except Exception as e:
    raise HTTPException(
        status_code=500, detail=f"Error processing query: {str(e)}"
    )


@app.get("/health", status_code=status.HTTP_200_OK)
def health_check():
  return {"status": "healthy", "service": "RAG API running"}

Start the Server

In [ ]:
import threading
import uvicorn


def run_server():
  uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")


server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("FastAPI server started! Go to http://127.0.0.1:8000/docs")

FastAPI server started in the background! Go to http://127.0.0.1:8001/docs


INFO:     Started server process [11268]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 10048] error while attempting to bind on address ('127.0.0.1', 8001): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
